In [ ]:
!pip install pycocotools

In [ ]:
import os
import zipfile
from pycocotools.coco import COCO
import requests
from tqdm import tqdm
from google.colab import drive

In [ ]:
# Mount Google Drive
drive.mount('/content/drive')

# Directory paths
drive_dir = '/content/drive/MyDrive/Colab Notebooks/coco_animal_images'
mask_drive_dir = '/content/drive/MyDrive/Colab Notebooks/mask_coco_animal_images'
os.makedirs(drive_dir, exist_ok=True)
os.makedirs(mask_drive_dir, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# COCO API URLs
annotations_url = 'http://images.cocodataset.org/annotations/annotations_trainval2017.zip'
images_train_url = 'http://images.cocodataset.org/zips/train2017.zip'

In [ ]:
# Download and extract annotations
annotations_zip_path = '/content/annotations_trainval2017.zip'
if not os.path.exists(annotations_zip_path):
    print("Downloading COCO annotations...")
    with open(annotations_zip_path, 'wb') as f:
        response = requests.get(annotations_url, stream=True)
        total_size = int(response.headers.get('content-length', 0))
        block_size = 1024
        for data in tqdm(response.iter_content(block_size), total=total_size // block_size, unit='KB', desc='Annotations'):
            f.write(data)

    print("Extracting COCO annotations...")
    with zipfile.ZipFile(annotations_zip_path, 'r') as zip_ref:
        zip_ref.extractall('/content')

Annotations: 246981KB [00:07, 34072.43KB/s]                            


Extracting COCO annotations...


In [ ]:
# Load COCO annotations for training set
coco_train = COCO('/content/annotations/instances_train2017.json')

# COCO categories containing animals
animal_categories = ['bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe']

loading annotations into memory...
Done (t=38.75s)
creating index...
index created!


In [ ]:
# Get image ids containing animals
animal_image_ids_train = []

for category in animal_categories:
    cat_ids_train = coco_train.getCatIds(catNms=category)
    img_ids_train = coco_train.getImgIds(catIds=cat_ids_train)
    animal_image_ids_train.extend(img_ids_train)

In [ ]:
# Download animal images for training set
for img_id in tqdm(animal_image_ids_train, desc='Downloading Train Images'):
    img_info = coco_train.loadImgs(img_id)[0]
    img_url = img_info['coco_url']
    img_path = os.path.join(drive_dir, img_info['file_name'])

    if not os.path.exists(img_path):
        img_data = requests.get(img_url).content
        with open(img_path, 'wb') as img_file:
            img_file.write(img_data)

print("Download complete. Images saved to Google Drive.")

Download complete. Images saved to Google Drive.
